In [7]:
from optclaw.config import get_app_config
from optclaw.models import create_chat_model
from langchain.agents import create_agent

In [8]:
from optclaw.agents.thread_state import ThreadDataState
from optclaw.agents.middlewares.thread_data_middleware import ThreadDataMiddleware
from optclaw.agents.middlewares.uploads_middleware import UploadsMiddleware

In [9]:
from optclaw.skills import get_skills_root_path
get_skills_root_path()

WindowsPath('E:/智能体/optclaw/skills')

In [4]:
path1 = '/mnt/skills/public/surprise-me/SKILL.md'
path2 = '/mnt/user-data/'

In [5]:
from optclaw.config import get_paths, get_app_config

# 1. 只获取一次配置（安全、高效）
paths = get_paths()
app_config = get_app_config()

# 2. 基础目录
root_dir = paths.base_dir.parent.parent
print(root_dir)

# 3. 容器路径前缀
container_prefix = app_config.skills.container_path

# 4. 安全切割路径（确保以 prefix 开头）
if path1.startswith(container_prefix):
    relative_path = path1[len(container_prefix):]
else:
    relative_path = path1  # 兜底，避免错误

# 5. 拼接最终路径（最安全）
full_path = root_dir / "skills" / relative_path.lstrip("/aas")  # 去掉开头斜杠，避免路径错乱

print(full_path.resolve())

In [6]:
container_skill_path = get_app_config().skills.container_path
actual_path = get_paths().base_dir.parent.parent / "skills" / path1[len(container_skill_path):].lstrip("/")
actual_path

WindowsPath('E:/智能体/optclaw/skills/public/surprise-me/SKILL.md')

In [7]:
import os
is_readonly = not os.access(full_path, os.W_OK)
is_readonly

True

In [8]:
from optclaw.tools.builtins import write_file_tool

In [ ]:
if os.path.exists(actual_path) and not os.access(actual_path, os.W_OK):
    print(f"Path {actual_path} is read-only.")

Path E:\智能体\optclaw\skills\public\surprise-me\SKILL.md is read-only.


In [11]:
path4 = '/mnt/skills/public/surprise-me/test.md'

In [12]:
write_file_tool(path4, "Hello, OptClaw!")

'Hello, OptClaw!'

# SKILL(本质是提示词)

In [22]:
# 设置技能文件开启标记，加载哪些技能基本信息
from optclaw.config import get_extensions_config
get_extensions_config()

ExtensionsConfig(mcp_servers={'filesystem': McpServerConfig(enabled=False, type='stdio', command='npx', args=['-y', '@modelcontextprotocol/server-filesystem', '/path/to/allowed/files'], env={}, url=None, headers={}, oauth=None, description='Provides filesystem access within allowed directories'), 'github': McpServerConfig(enabled=False, type='stdio', command='npx', args=['-y', '@modelcontextprotocol/server-github'], env={'GITHUB_TOKEN': ''}, url=None, headers={}, oauth=None, description='GitHub MCP server for repository operations'), 'postgres': McpServerConfig(enabled=False, type='stdio', command='npx', args=['-y', '@modelcontextprotocol/server-postgres', 'postgresql://localhost/mydb'], env={}, url=None, headers={}, oauth=None, description='PostgreSQL database access')}, skills={'academic-paper-review': SkillStateConfig(enabled=False)})

In [23]:
# 技能加载
from optclaw.agents.pormpt_manager import prime_enabled_skills_cache
prime_enabled_skills_cache()

In [24]:
from optclaw.agents.pormpt_manager import get_skills_prompt_section,_get_enabled_skills

In [25]:
get_skills_prompt_section()

'<skill_system>\nYou have access to skills that provide optimized workflows for specific tasks. Each skill contains best practices, frameworks, and references to additional resources.\n\n**Progressive Loading Pattern:**\n1. When a user query matches a skill\'s use case, immediately call `read_file` on the skill\'s main file using the path attribute provided in the skill tag below\n2. Read and understand the skill\'s workflow and instructions\n3. The skill file contains references to external resources under the same folder\n4. Load referenced resources only when needed during execution\n5. Follow the skill\'s instructions precisely\n\n**Skills are located at:** /mnt/skills\n\n<available_skills>\n    <skill>\n        <name>bootstrap</name>\n        <description>Generate a personalized SOUL.md through a warm, adaptive onboarding conversation. Trigger when the user wants to create, set up, or initialize their AI partner\'s identity — e.g., "create my SOUL.md", "bootstrap my agent", "set u

# 工具（mcp+tools）

In [1]:
from optclaw.tools import get_available_tools
tools = get_available_tools()

format='2026-05-12 11:29:00,913 - optclaw.tools.tools - get_available_tools - INFO - Tool configs after group filtering: ['web_search', 'web_fetch', 'image_search']'
format='2026-05-12 11:29:01,036 - optclaw.tools.tools - get_available_tools - INFO - Total tools loaded: 3, built-in tools: 6'


In [2]:
tools

[StructuredTool(name='web_search', description='Search the web for information. Use this tool to find current information, news, articles, and facts from the internet.', args_schema=<class 'langchain_core.utils.pydantic.web_search'>, func=<function web_search_tool at 0x00000186B5879620>),
 StructuredTool(name='web_fetch', description='Fetch the contents of a web page at a given URL.\nOnly fetch EXACT URLs that have been provided directly by the user or have been returned in results from the web_search and web_fetch tools.\nThis tool can NOT access content that requires authentication, such as private Google Docs or pages behind login walls.\nDo NOT add www. to URLs that do NOT have them.\nURLs must include the schema: https://example.com is a valid URL while example.com is an invalid URL.', args_schema=<class 'langchain_core.utils.pydantic.web_fetch'>, coroutine=<function web_fetch_tool at 0x00000186B5879C60>),
 StructuredTool(name='image_search', description='Search for images online.

In [1]:
path4 = '/mnt/skills/public/surprise-me/SKILL.md'

In [2]:
from optclaw.tools.builtins import grep_file_tool,glob_file_tool

In [3]:
glob_file_tool(path4)

"Found 1 items matching pattern '/mnt/skills/public/surprise-me/':\n- surprise-me"

In [6]:
grep_file_tool(path4, "Step")

"Found 5 matches for 'Step' in /mnt/skills/public/surprise-me/SKILL.md:\nLine 12: ### Step 1: Discover Available Skills\nLine 16: ### Step 2: Plan the Surprise\nLine 33: ### Step 3: Fallback — No Other Skills Available\nLine 41: ### Step 4: Execute\nLine 48: ### Step 5: Reveal"

In [14]:
get_app_config().get_tool_config("web_search")

ToolConfig(name='web_search', group='web', use='optclaw.community.ddg_search.tools:web_search_tool', max_results=5)

# Agent

In [1]:
"""OptClawClient — Embedded Python client for optclaw agent system.

Provides direct programmatic access to optclaw's agent capabilities
without requiring LangGraph Server or Gateway API processes.

Usage:
    from optclaw.client import OptClawClient

    client = OptClawClient()
    response = client.chat("Analyze this paper for me", thread_id="my-thread")
    print(response)

    # Streaming
    for event in client.stream("hello"):
        print(event)
"""

import asyncio
import json
import logging
import mimetypes
import shutil
import tempfile
import uuid
from collections.abc import Generator, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Literal

from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.runnables import RunnableConfig

from optclaw.agents import apply_prompt_template
from optclaw.agents.thread_state import ThreadState
from optclaw.config.agents_config import AGENT_NAME_PATTERN
from optclaw.config.app_config import get_app_config, reload_app_config
# from optclaw.config.extensions_config import ExtensionsConfig, SkillStateConfig, get_extensions_config, reload_extensions_config
from optclaw.config.paths import get_paths
from optclaw.models import create_chat_model
# from optclaw.skills.installer import install_skill_from_archive
from optclaw.uploads.manager import (
    claim_unique_filename,
    delete_file_safe,
    enrich_file_listing,
    ensure_uploads_dir,
    get_uploads_dir,
    list_files_in_dir,
    upload_artifact_url,
    upload_virtual_path,
)
from optclaw.agents.middlewares import build_middlewares

from optclaw.log import setup_logging
logger = setup_logging(__name__)


StreamEventType = Literal["values", "messages-tuple", "custom", "end"]

In [2]:
@dataclass
class StreamEvent:
    """A single event from the streaming agent response.

    Event types align with the LangGraph SSE protocol:
        - ``"values"``: Full state snapshot (title, messages, artifacts).
        - ``"messages-tuple"``: Per-message update (AI text, tool calls, tool results).
        - ``"end"``: Stream finished.

    Attributes:
        type: Event type.
        data: Event payload. Contents vary by type.
    """

    type: StreamEventType
    data: dict[str, Any] = field(default_factory=dict)

In [3]:
class OptClawClient:
    """Embedded Python client for optclaw agent system.

    Provides direct programmatic access to optclaw's agent capabilities
    without requiring LangGraph Server or Gateway API processes.

    Note:
        Multi-turn conversations require a ``checkpointer``. Without one,
        each ``stream()`` / ``chat()`` call is stateless — ``thread_id``
        is only used for file isolation (uploads / artifacts).

        The system prompt (including date, memory, and skills context) is
        generated when the internal agent is first created and cached until
        the configuration key changes. Call :meth:`reset_agent` to force
        a refresh in long-running processes.

    Example::

        from optclaw.client import OptClawClient

        client = OptClawClient()

        # Simple one-shot
        print(client.chat("hello"))

        # Streaming
        for event in client.stream("hello"):
            print(event.type, event.data)

        # Configuration queries
        print(client.list_models())
        print(client.list_skills())
    """

    def __init__(
        self,
        config_path: str | None = None,
        checkpointer = None,
        *,
        model_name: str | None = None,
        thinking_enabled: bool = False,
        subagent_enabled: bool = False,
        plan_mode: bool = True,
        agent_name: str | None = None,
        available_skills: set[str] | None = None,
        middlewares: Sequence[AgentMiddleware] | None = None,
    ):
        """Initialize the client.

        Loads configuration but defers agent creation to first use.

        Args:
            config_path: Path to config.yaml. Uses default resolution if None.
            checkpointer: LangGraph checkpointer instance for state persistence.
                Required for multi-turn conversations on the same thread_id.
                Without a checkpointer, each call is stateless.
            model_name: Override the default model name from config.
            thinking_enabled: Enable model's extended thinking.
            subagent_enabled: Enable subagent delegation.
            plan_mode: Enable TodoList middleware for plan mode.
            agent_name: Name of the agent to use.
            available_skills: Optional set of skill names to make available. If None (default), all scanned skills are available.
            middlewares: Optional list of custom middlewares to inject into the agent.
        """
        if config_path is not None:
            reload_app_config(config_path)
        self._app_config = get_app_config()

        if agent_name is not None and not AGENT_NAME_PATTERN.match(agent_name):
            raise ValueError(f"Invalid agent name '{agent_name}'. Must match pattern: {AGENT_NAME_PATTERN.pattern}")

        self._checkpointer = checkpointer
        self._model_name = model_name
        self._thinking_enabled = thinking_enabled
        self._subagent_enabled = subagent_enabled
        self._plan_mode = plan_mode
        self._agent_name = agent_name
        self._available_skills = set(available_skills) if available_skills is not None else None
        self._middlewares = list(middlewares) if middlewares else []

        # Lazy agent — created on first call, recreated when config changes.
        self._agent = None
        self._agent_config_key: tuple | None = None
    def reset_agent(self) -> None:
        """Force the internal agent to be recreated on the next call.

        Use this after external changes (e.g. memory updates, skill
        installations) that should be reflected in the system prompt
        or tool set.
        """
        self._agent = None
        self._agent_config_key = None

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    @staticmethod
    def _atomic_write_json(path: Path, data: dict) -> None:
        """Write JSON to *path* atomically (temp file + replace)."""
        fd = tempfile.NamedTemporaryFile(
            mode="w",
            dir=path.parent,
            suffix=".tmp",
            delete=False,
        )
        try:
            json.dump(data, fd, indent=2)
            fd.close()
            Path(fd.name).replace(path)
        except BaseException:
            fd.close()
            Path(fd.name).unlink(missing_ok=True)
            raise

    def _get_runnable_config(self, thread_id: str, **overrides) -> RunnableConfig:
        """Build a RunnableConfig for agent invocation."""
        configurable = {
            "thread_id": thread_id,
            "model_name": overrides.get("model_name", self._model_name),
            "thinking_enabled": overrides.get("thinking_enabled", self._thinking_enabled),
            "is_plan_mode": overrides.get("plan_mode", self._plan_mode),
            "subagent_enabled": overrides.get("subagent_enabled", self._subagent_enabled),
        }
        return RunnableConfig(
            configurable=configurable,
            recursion_limit=overrides.get("recursion_limit", 100),
        )

    def _ensure_agent(self, config: RunnableConfig):
        """Create (or recreate) the agent when config-dependent params change."""
        cfg = config.get("configurable", {})
        key = (
            cfg.get("model_name"),
            cfg.get("thinking_enabled"),
            cfg.get("is_plan_mode"),
            cfg.get("subagent_enabled"),
            self._agent_name,
            frozenset(self._available_skills) if self._available_skills is not None else None,
        )

        if self._agent is not None and self._agent_config_key == key:
            return

        thinking_enabled = cfg.get("thinking_enabled", True)
        model_name = cfg.get("model_name")
        subagent_enabled = cfg.get("subagent_enabled", False)
        max_concurrent_subagents = cfg.get("max_concurrent_subagents", 3)

        kwargs: dict[str, Any] = {
            "model": create_chat_model(name=model_name, thinking_enabled=thinking_enabled),
            "tools": self._get_tools(model_name=model_name, subagent_enabled=subagent_enabled),
            "middleware": build_middlewares(name=model_name, plan_mode=self._plan_mode),# _build_middlewares(config, model_name=model_name, agent_name=self._agent_name, custom_middlewares=self._middlewares),
            "system_prompt": apply_prompt_template(
                agent_name=self._agent_name,
                available_skills=self._available_skills,
                subagent_enabled=subagent_enabled,
                max_concurrent_subagents=max_concurrent_subagents
            ),
            "state_schema": ThreadState,  # record state info during the chat
            # "context_schema": None  # provide context info such as user id, read-only mode
        }
        checkpointer = self._checkpointer
        if checkpointer is None:
            from optclaw.agents.checkpointer import get_checkpointer

            checkpointer = get_checkpointer()

        if checkpointer is not None:
            kwargs["checkpointer"] = checkpointer

        self._agent = create_agent(**kwargs)
        self._agent_config_key = key
        logger.info("Agent created: agent_name=%s, model=%s, thinking=%s", self._agent_name, model_name, thinking_enabled)

    @staticmethod
    def _get_tools(*, model_name: str | None, subagent_enabled: bool):
        """Lazy import to avoid circular dependency at module level."""
        from optclaw.tools import get_available_tools

        return get_available_tools(model_name=model_name, subagent_enabled=subagent_enabled)

    @staticmethod
    def _serialize_tool_calls(tool_calls) -> list[dict]:
        """Reshape LangChain tool_calls into the wire format used in events."""
        return [{"name": tc["name"], "args": tc["args"], "id": tc.get("id")} for tc in tool_calls]

    @staticmethod
    def _ai_text_event(msg_id: str | None, text: str, usage: dict | None) -> "StreamEvent":
        """Build a ``messages-tuple`` AI text event, attaching usage when present."""
        data: dict[str, Any] = {"type": "ai", "content": text, "id": msg_id}
        if usage:
            data["usage_metadata"] = usage
        return StreamEvent(type="messages-tuple", data=data)

    @staticmethod
    def _ai_tool_calls_event(msg_id: str | None, tool_calls) -> "StreamEvent":
        """Build a ``messages-tuple`` AI tool-calls event."""
        return StreamEvent(
            type="messages-tuple",
            data={
                "type": "ai",
                "content": "",
                "id": msg_id,
                "tool_calls": OptClawClient._serialize_tool_calls(tool_calls),
            },
        )

    @staticmethod
    def _tool_message_event(msg: ToolMessage) -> "StreamEvent":
        """Build a ``messages-tuple`` tool-result event from a ToolMessage."""
        return StreamEvent(
            type="messages-tuple",
            data={
                "type": "tool",
                "content": OptClawClient._extract_text(msg.content),
                "name": msg.name,
                "tool_call_id": msg.tool_call_id,
                "id": msg.id,
            },
        )

    @staticmethod
    def _serialize_message(msg) -> dict:
        """Serialize a LangChain message to a plain dict for values events."""
        if isinstance(msg, AIMessage):
            d: dict[str, Any] = {"type": "ai", "content": msg.content, "id": getattr(msg, "id", None)}
            if msg.tool_calls:
                d["tool_calls"] = OptClawClient._serialize_tool_calls(msg.tool_calls)
            if getattr(msg, "usage_metadata", None):
                d["usage_metadata"] = msg.usage_metadata
            return d
        if isinstance(msg, ToolMessage):
            return {
                "type": "tool",
                "content": OptClawClient._extract_text(msg.content),
                "name": getattr(msg, "name", None),
                "tool_call_id": getattr(msg, "tool_call_id", None),
                "id": getattr(msg, "id", None),
            }
        if isinstance(msg, HumanMessage):
            return {"type": "human", "content": msg.content, "id": getattr(msg, "id", None)}
        if isinstance(msg, SystemMessage):
            return {"type": "system", "content": msg.content, "id": getattr(msg, "id", None)}
        return {"type": "unknown", "content": str(msg), "id": getattr(msg, "id", None)}

    @staticmethod
    def _extract_text(content) -> str:
        """Extract plain text from AIMessage content (str or list of blocks).

        String chunks are concatenated without separators to avoid corrupting
        token/character deltas or chunked JSON payloads. Dict-based text blocks
        are treated as full text blocks and joined with newlines to preserve
        readability.
        """
        if isinstance(content, str):
            return content
        if isinstance(content, list):
            if content and all(isinstance(block, str) for block in content):
                chunk_like = len(content) > 1 and all(isinstance(block, str) and len(block) <= 20 and any(ch in block for ch in '{}[]":,') for block in content)
                return "".join(content) if chunk_like else "\n".join(content)

            pieces: list[str] = []
            pending_str_parts: list[str] = []

            def flush_pending_str_parts() -> None:
                if pending_str_parts:
                    pieces.append("".join(pending_str_parts))
                    pending_str_parts.clear()

            for block in content:
                if isinstance(block, str):
                    pending_str_parts.append(block)
                elif isinstance(block, dict):
                    flush_pending_str_parts()
                    text_val = block.get("text")
                    if isinstance(text_val, str):
                        pieces.append(text_val)

            flush_pending_str_parts()
            return "\n".join(pieces) if pieces else ""
        return str(content)
    
    # ------------------------------------------------------------------
    # Public API — threads
    # ------------------------------------------------------------------
    def list_threads(self, limit: int = 10) -> dict:
        """List the recent N threads.

        Args:
            limit: Maximum number of threads to return. Default is 10.

        Returns:
            Dict with "thread_list" key containing list of thread info dicts,
            sorted by thread creation time descending.
        """
        checkpointer = self._checkpointer
        if checkpointer is None:
            from optclaw.agents.checkpointer.provider import get_checkpointer

            checkpointer = get_checkpointer()

        thread_info_map = {}

        for cp in checkpointer.list(config=None, limit=limit):
            cfg = cp.config.get("configurable", {})
            thread_id = cfg.get("thread_id")
            if not thread_id:
                continue

            ts = cp.checkpoint.get("ts")
            checkpoint_id = cfg.get("checkpoint_id")

            if thread_id not in thread_info_map:
                channel_values = cp.checkpoint.get("channel_values", {})
                thread_info_map[thread_id] = {
                    "thread_id": thread_id,
                    "created_at": ts,
                    "updated_at": ts,
                    "latest_checkpoint_id": checkpoint_id,
                    "title": channel_values.get("title"),
                }
            else:
                # Explicitly compare timestamps to ensure accuracy when iterating over unordered namespaces.
                # Treat None as "missing" and only compare when existing values are non-None.
                if ts is not None:
                    current_created = thread_info_map[thread_id]["created_at"]
                    if current_created is None or ts < current_created:
                        thread_info_map[thread_id]["created_at"] = ts

                    current_updated = thread_info_map[thread_id]["updated_at"]
                    if current_updated is None or ts > current_updated:
                        thread_info_map[thread_id]["updated_at"] = ts
                        thread_info_map[thread_id]["latest_checkpoint_id"] = checkpoint_id
                        channel_values = cp.checkpoint.get("channel_values", {})
                        thread_info_map[thread_id]["title"] = channel_values.get("title")

        threads = list(thread_info_map.values())
        threads.sort(key=lambda x: x.get("created_at") or "", reverse=True)

        return {"thread_list": threads[:limit]}

    def get_thread(self, thread_id: str) -> dict:
        """Get the complete thread record, including all node execution records.

        Args:
            thread_id: Thread ID.

        Returns:
            Dict containing the thread's full checkpoint history.
        """
        checkpointer = self._checkpointer
        if checkpointer is None:
            from optclaw.agents.checkpointer.provider import get_checkpointer

            checkpointer = get_checkpointer()

        config = {"configurable": {"thread_id": thread_id}}
        checkpoints = []

        for cp in checkpointer.list(config):
            channel_values = dict(cp.checkpoint.get("channel_values", {}))
            if "messages" in channel_values:
                channel_values["messages"] = [self._serialize_message(m) if hasattr(m, "content") else m for m in channel_values["messages"]]

            cfg = cp.config.get("configurable", {})
            parent_cfg = cp.parent_config.get("configurable", {}) if cp.parent_config else {}

            checkpoints.append(
                {
                    "checkpoint_id": cfg.get("checkpoint_id"),
                    "parent_checkpoint_id": parent_cfg.get("checkpoint_id"),
                    "ts": cp.checkpoint.get("ts"),
                    "metadata": cp.metadata,
                    "values": channel_values,
                    "pending_writes": [{"task_id": w[0], "channel": w[1], "value": w[2]} for w in getattr(cp, "pending_writes", [])],
                }
            )

        # Sort globally by timestamp to prevent partial ordering issues caused by different namespaces (e.g., subgraphs)
        checkpoints.sort(key=lambda x: x["ts"] if x["ts"] else "")

        return {"thread_id": thread_id, "checkpoints": checkpoints}
    
    # ------------------------------------------------------------------
    # Public API — conversation
    # ------------------------------------------------------------------

    def stream(
        self,
        message: str,
        *,
        thread_id: str | None = None,
        **kwargs,
    ) -> Generator[StreamEvent, None, None]:
        """Stream a conversation turn, yielding events incrementally.

        Each call sends one user message and yields events until the agent
        finishes its turn. A ``checkpointer`` must be provided at init time
        for multi-turn context to be preserved across calls.

        Event types align with the LangGraph SSE protocol so that
        consumers can switch between HTTP streaming and embedded mode
        without changing their event-handling logic.

        Token-level streaming
        ~~~~~~~~~~~~~~~~~~~~~
        This method subscribes to LangGraph's ``messages`` stream mode, so
        ``messages-tuple`` events for AI text are emitted as **deltas** as
        the model generates tokens, not as one cumulative dump at node
        completion.  Each delta carries a stable ``id`` — consumers that
        want the full text must accumulate ``content`` per ``id``.
        ``chat()`` already does this for you.

        Tool calls and tool results are still emitted once per logical
        message.  ``values`` events continue to carry full state snapshots
        after each graph node finishes; AI text already delivered via the
        ``messages`` stream is **not** re-synthesized from the snapshot to
        avoid duplicate deliveries.

        Why not reuse Gateway's ``run_agent``?
        ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
        Gateway (``runtime/runs/worker.py``) has a complete streaming
        pipeline: ``run_agent`` → ``StreamBridge`` → ``sse_consumer``.  It
        looks like this client duplicates that work, but the two paths
        serve different audiences and **cannot** share execution:

        * ``run_agent`` is ``async def`` and uses ``agent.astream()``;
          this method is a sync generator using ``agent.stream()`` so
          callers can write ``for event in client.stream(...)`` without
          touching asyncio.  Bridging the two would require spinning up
          an event loop + thread per call.
        * Gateway events are JSON-serialized by ``serialize()`` for SSE
          wire transmission.  This client yields in-process stream event
          payloads directly as Python data structures (``StreamEvent``
          with ``data`` as a plain ``dict``), without the extra
          JSON/SSE serialization layer used for HTTP delivery.
        * ``StreamBridge`` is an asyncio-queue decoupling producers from
          consumers across an HTTP boundary (``Last-Event-ID`` replay,
          heartbeats, multi-subscriber fan-out).  A single in-process
          caller with a direct iterator needs none of that.

        So ``OptClawClient.stream()`` is a parallel, sync, in-process
        consumer of the same ``create_agent()`` factory — not a wrapper
        around Gateway.  The two paths **should** stay in sync on which
        LangGraph stream modes they subscribe to; that invariant is
        enforced by ``tests/test_client.py::test_messages_mode_emits_token_deltas``
        rather than by a shared constant, because the three layers
        (Graph, Platform SDK, HTTP) each use their own naming
        (``messages`` vs ``messages-tuple``) and cannot literally share
        a string.

        Args:
            message: User message text.
            thread_id: Thread ID for conversation context. Auto-generated if None.
            **kwargs: Override client defaults (model_name, thinking_enabled,
                plan_mode, subagent_enabled, recursion_limit).

        Yields:
            StreamEvent with one of:
            - type="values"          data={"title": str|None, "messages": [...], "artifacts": [...]}
            - type="custom"          data={...}
            - type="messages-tuple"  data={"type": "ai", "content": <delta>, "id": str}
            - type="messages-tuple"  data={"type": "ai", "content": <delta>, "id": str, "usage_metadata": {...}}
            - type="messages-tuple"  data={"type": "ai", "content": "", "id": str, "tool_calls": [...]}
            - type="messages-tuple"  data={"type": "tool", "content": str, "name": str, "tool_call_id": str, "id": str}
            - type="end"             data={"usage": {"input_tokens": int, "output_tokens": int, "total_tokens": int}}
        """
        if thread_id is None:
            thread_id = str(uuid.uuid4())

        config = self._get_runnable_config(thread_id, **kwargs)
        self._ensure_agent(config)

        state: dict[str, Any] = {"messages": [HumanMessage(content=message)]}
        context = {"thread_id": thread_id}
        if self._agent_name:
            context["agent_name"] = self._agent_name

        seen_ids: set[str] = set()
        # Cross-mode handoff: ids already streamed via LangGraph ``messages``
        # mode so the ``values`` path skips re-synthesis of the same message.
        streamed_ids: set[str] = set()
        # The same message id carries identical cumulative ``usage_metadata``
        # in both the final ``messages`` chunk and the values snapshot —
        # count it only on whichever arrives first.
        counted_usage_ids: set[str] = set()
        cumulative_usage: dict[str, int] = {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0}

        def _account_usage(msg_id: str | None, usage: Any) -> dict | None:
            """Add *usage* to cumulative totals if this id has not been counted.

            ``usage`` is a ``langchain_core.messages.UsageMetadata`` TypedDict
            or ``None``; typed as ``Any`` because TypedDicts are not
            structurally assignable to plain ``dict`` under strict type
            checking.  Returns the normalized usage dict (for attaching
            to an event) when we accepted it, otherwise ``None``.
            """
            if not usage:
                return None
            if msg_id and msg_id in counted_usage_ids:
                return None
            if msg_id:
                counted_usage_ids.add(msg_id)
            input_tokens = usage.get("input_tokens", 0) or 0
            output_tokens = usage.get("output_tokens", 0) or 0
            total_tokens = usage.get("total_tokens", 0) or 0
            cumulative_usage["input_tokens"] += input_tokens
            cumulative_usage["output_tokens"] += output_tokens
            cumulative_usage["total_tokens"] += total_tokens
            return {
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "total_tokens": total_tokens,
            }

        for item in self._agent.stream(
            state,
            config=config,
            context=context,
            stream_mode=["values", "messages", "custom"],
        ):
            if isinstance(item, tuple) and len(item) == 2:
                mode, chunk = item
                mode = str(mode)
            else:
                mode, chunk = "values", item

            if mode == "custom":
                yield StreamEvent(type="custom", data=chunk)
                continue

            if mode == "messages":
                # LangGraph ``messages`` mode emits ``(message_chunk, metadata)``.
                if isinstance(chunk, tuple) and len(chunk) == 2:
                    msg_chunk, _metadata = chunk
                else:
                    msg_chunk = chunk

                msg_id = getattr(msg_chunk, "id", None)

                if isinstance(msg_chunk, AIMessage):
                    text = self._extract_text(msg_chunk.content)
                    counted_usage = _account_usage(msg_id, msg_chunk.usage_metadata)

                    if text:
                        if msg_id:
                            streamed_ids.add(msg_id)
                        yield self._ai_text_event(msg_id, text, counted_usage)

                    if msg_chunk.tool_calls:
                        if msg_id:
                            streamed_ids.add(msg_id)
                        yield self._ai_tool_calls_event(msg_id, msg_chunk.tool_calls)

                elif isinstance(msg_chunk, ToolMessage):
                    if msg_id:
                        streamed_ids.add(msg_id)
                    yield self._tool_message_event(msg_chunk)
                continue

            # mode == "values"
            messages = chunk.get("messages", [])

            for msg in messages:
                msg_id = getattr(msg, "id", None)
                if msg_id and msg_id in seen_ids:
                    continue
                if msg_id:
                    seen_ids.add(msg_id)

                # Already streamed via ``messages`` mode; only (defensively)
                # capture usage here and skip re-synthesizing the event.
                if msg_id and msg_id in streamed_ids:
                    if isinstance(msg, AIMessage):
                        _account_usage(msg_id, getattr(msg, "usage_metadata", None))
                    continue

                if isinstance(msg, AIMessage):
                    counted_usage = _account_usage(msg_id, msg.usage_metadata)

                    if msg.tool_calls:
                        yield self._ai_tool_calls_event(msg_id, msg.tool_calls)

                    text = self._extract_text(msg.content)
                    if text:
                        yield self._ai_text_event(msg_id, text, counted_usage)

                elif isinstance(msg, ToolMessage):
                    yield self._tool_message_event(msg)

            # Emit a values event for each state snapshot
            yield StreamEvent(
                type="values",
                data={
                    "title": chunk.get("title"),
                    "messages": [self._serialize_message(m) for m in messages],
                    "artifacts": chunk.get("artifacts", []),
                },
            )

        yield StreamEvent(type="end", data={"usage": cumulative_usage})

    def chat(self, message: str, *, thread_id: str | None = None, **kwargs) -> str:
        """Send a message and return the final text response.

        Convenience wrapper around :meth:`stream` that accumulates delta
        ``messages-tuple`` events per ``id`` and returns the text of the
        **last** AI message to complete.  Intermediate AI messages (e.g.
        planner drafts) are discarded — only the final id's accumulated
        text is returned.  Use :meth:`stream` directly if you need every
        delta as it arrives.

        Args:
            message: User message text.
            thread_id: Thread ID for conversation context. Auto-generated if None.
            **kwargs: Override client defaults (same as stream()).

        Returns:
            The accumulated text of the last AI message, or empty string
            if no AI text was produced.
        """
        # Per-id delta lists joined once at the end — avoids the O(n²) cost
        # of repeated ``str + str`` on a growing buffer for long responses.
        chunks: dict[str, list[str]] = {}
        last_id: str = ""
        for event in self.stream(message, thread_id=thread_id, **kwargs):
            if event.type == "messages-tuple" and event.data.get("type") == "ai":
                msg_id = event.data.get("id") or ""
                delta = event.data.get("content", "")
                if delta:
                    chunks.setdefault(msg_id, []).append(delta)
                    last_id = msg_id
        return "".join(chunks.get(last_id, ()))

In [4]:
occ = OptClawClient()

In [ ]:
occ.chat("你是谁")

format='2026-05-12 22:34:24,574 - optclaw.tools.tools - get_available_tools - INFO - Tool configs after group filtering: ['web_search', 'web_fetch', 'image_search']'
format='2026-05-12 22:34:24,731 - optclaw.tools.tools - get_available_tools - INFO - Including view_image_tool for model 'doubao-seed-1.8' (supports_vision=True)'
format='2026-05-12 22:34:24,732 - optclaw.tools.tools - get_available_tools - INFO - Total tools loaded: 3, built-in tools: 7'
format='2026-05-12 22:34:24,894 - optclaw.agents.checkpointer.provider - _sync_checkpointer_cm - INFO - Checkpointer: using SqliteSaver (E:\智能体\optclaw\optclaw\.opt-claw\checkpoints.db)'
format='2026-05-12 22:34:24,914 - __main__ - _ensure_agent - INFO - Agent created: agent_name=None, model=None, thinking=False'
format='2026-05-12 22:34:26,697 - httpx - _send_single_request - INFO - HTTP Request: POST https://ark.cn-beijing.volces.com/api/v3/chat/completions "HTTP/1.1 200 OK"'
format='2026-05-12 22:34:29,705 - httpx - _send_single_reques

'你好！👋 我是你的AI协作伙伴。为了更好地了解彼此，我们可以从简单的开始：\n\n你平时更习惯用中文还是英文交流呢？ 😊'

format='2026-05-12 22:35:03,574 - optclaw.agents.memory.queue - _process_queue - INFO - Processing 1 queued memory updates'
format='2026-05-12 22:35:03,576 - optclaw.agents.memory.queue - _process_queue - INFO - Updating memory for thread eaf67e33-d0b2-4f7d-a199-b7af8d0b3827'


In [5]:
# for data in occ.stream(""):
#     print(data)

In [7]:
from optclaw.agents.pormpt_manager import apply_prompt_template
apply_prompt_template(
    agent_name="zpf",
    available_skills=['bootstrap'])

E:\智能体\optclaw\optclaw\.opt-claw\agents\zpf\memory.json


'\n<role>\nYou are zpf, an open-source super agent.\n</role>\n\n\n\n\n<thinking_style>\n- Think concisely and strategically about the user\'s request BEFORE taking action\n- Break down the task: What is clear? What is ambiguous? What is missing?\n- **PRIORITY CHECK: If anything is unclear, missing, or has multiple interpretations, you MUST ask for clarification FIRST - do NOT proceed with work**\n</thinking_style>\n\n<clarification_system>\n**WORKFLOW PRIORITY: CLARIFY → PLAN → ACT**\n1. **FIRST**: Analyze the request in your thinking - identify what\'s unclear, missing, or ambiguous\n2. **SECOND**: If clarification is needed, call `ask_clarification` tool IMMEDIATELY - do NOT start working\n3. **THIRD**: Only after all clarifications are resolved, proceed with planning and execution\n\n**CRITICAL RULE: Clarification ALWAYS comes BEFORE action. Never start working and clarify mid-execution.**\n\n**MANDATORY Clarification Scenarios - You MUST call ask_clarification BEFORE starting work 

In [8]:
from optclaw.config import get_app_config

config = get_app_config()
print(config.skills.get_skills_path())

E:\智能体\optclaw\skills


In [6]:
config.get_model_config(name="optclaw-v3")

In [ ]:
# if thread_id is None:
#     thread_id = str(uuid.uuid4())

# config = self._get_runnable_config(thread_id, **kwargs)
# self._ensure_agent(config)

# state: dict[str, Any] = {"messages": [HumanMessage(content=message)]}
# context = {"thread_id": thread_id}
# if self._agent_name:
#     context["agent_name"] = self._agent_name

In [ ]:
c = get_app_config()
model = create_chat_model(thinking_enabled=True)
# 创建Agent
agent = create_agent(
    model=model,
    system_prompt="你好"
)

# 正确调用
response = agent.invoke({
    "messages": [{"role": "user", "content": "今天天气怎么样？"}]
})

print(response)

In [ ]:
from optclaw.agents.thread_state import ThreadDataState
from optclaw.agents.middlewares.thread_data_middleware import ThreadDataMiddleware
from optclaw.agents.middlewares.uploads_middleware import UploadsMiddleware

None
ssssdddd


In [2]:
from optclaw.agents.pormpt_manager import prime_enabled_skills_cache
prime_enabled_skills_cache()

In [2]:
from optclaw.skills import get_skills_root_path

In [1]:
from optclaw.config import get_extensions_config

In [2]:
get_extensions_config()

ExtensionsConfig(mcp_servers={'filesystem': McpServerConfig(enabled=False, type='stdio', command='npx', args=['-y', '@modelcontextprotocol/server-filesystem', '/path/to/allowed/files'], env={}, url=None, headers={}, oauth=None, description='Provides filesystem access within allowed directories'), 'github': McpServerConfig(enabled=False, type='stdio', command='npx', args=['-y', '@modelcontextprotocol/server-github'], env={'GITHUB_TOKEN': ''}, url=None, headers={}, oauth=None, description='GitHub MCP server for repository operations'), 'postgres': McpServerConfig(enabled=False, type='stdio', command='npx', args=['-y', '@modelcontextprotocol/server-postgres', 'postgresql://localhost/mydb'], env={}, url=None, headers={}, oauth=None, description='PostgreSQL database access')}, skills={})

In [5]:
from optclaw.config import Paths

In [6]:
d = Paths()
d.base_dir

WindowsPath('E:/智能体/测试/optclaw/.opt-claw')

In [ ]:
!pip uninstall xquant

^C


In [1]:
!pip list

Package                                  Version
---------------------------------------- -------------------
agent-sandbox                            0.0.30
aiobotocore                              2.19.0
aiohappyeyeballs                         2.4.4
aiohttp                                  3.11.10
aioitertools                             0.7.1
aiosignal                                1.2.0
aiosqlite                                0.22.1
alabaster                                0.7.16
altair                                   5.5.0
anaconda-anon-usage                      0.7.1
anaconda-auth                            0.8.6
anaconda-catalogs                        0.2.0
anaconda-cli-base                        0.5.2
anaconda-client                          1.13.0
anaconda-navigator                       2.6.6
anaconda-project                         0.11.1
annotated-types                          0.6.0
anthropic                                0.84.0
anyio                              